<a href="https://colab.research.google.com/github/mromina24/PR24BKRMKPMSJSV/blob/main/596715_IA161_2026_Language_modelling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# IA161 Language modelling

The notebook is adapted from an old version of [minGPT](https://github.com/karpathy/minGPT).

## Train a character-level GPT on some text data

The inputs here are simple text files, which we chop up to individual characters and then train GPT on.  In this example we will feed it some Čapek or Shakespeare, which we'll get it to predict character-level.


In [ ]:
!wget https://www.fi.muni.cz/~pary/mingpt.zip
!unzip mingpt.zip

--2026-09-21 07:09:24--  https://www.fi.muni.cz/~pary/mingpt.zip
Resolving www.fi.muni.cz (www.fi.muni.cz)... 147.251.48.1, 2001:718:801:230::1
Connecting to www.fi.muni.cz (www.fi.muni.cz)|147.251.48.1|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 6072 (5.9K) [application/zip]
Saving to: ‘mingpt.zip’

mingpt.zip          100%[===================>]   5.93K  --.-KB/s    in 0s      

2026-09-21 07:09:26 (1.88 GB/s) - ‘mingpt.zip’ saved [6072/6072]

Archive:  mingpt.zip
 extracting: mingpt/__init__.py      
  inflating: mingpt/model.py         
  inflating: mingpt/trainer.py       
  inflating: mingpt/utils.py         


In [ ]:
# set up logging
import logging
logging.basicConfig(
        format="%(asctime)s - %(levelname)s - %(name)s -   %(message)s",
        datefmt="%m/%d/%Y %H:%M:%S",
        level=logging.INFO,
)

In [ ]:
# make deterministic
from mingpt.utils import set_seed
set_seed(42)

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.nn import functional as F

In [ ]:
import math
from torch.utils.data import Dataset

class CharDataset(Dataset):

    def __init__(self, data, block_size):
        chars = sorted(list(set(data)))
        data_size, vocab_size = len(data), len(chars)
        print('data has %d characters, %d unique.' % (data_size, vocab_size))

        self.stoi = { ch:i for i,ch in enumerate(chars) }
        self.itos = { i:ch for i,ch in enumerate(chars) }
        self.block_size = block_size
        self.vocab_size = vocab_size
        self.data = data

    def __len__(self):
        return len(self.data) - self.block_size

    def __getitem__(self, idx):
        # grab a chunk of (block_size + 1) characters from the data
        chunk = self.data[idx:idx + self.block_size + 1]
        # encode every character to an integer
        dix = [self.stoi[s] for s in chunk]
        """
        arrange data and targets so that the first i elements of x
        will be asked to predict the i-th element of y. Notice that
        the eventual language model will actually make block_size
        individual predictions at the same time based on this data,
        so we are being clever and amortizing the cost of the forward
        pass of the network. So for example if block_size is 4, then
        we could e.g. sample a chunk of text "hello", the integers in
        x will correspond to "hell" and in y will be "ello". This will
        then actually "multitask" 4 separate examples at the same time
        in the language model:
        - given just "h", please predict "e" as next
        - given "he" please predict "l" next
        - given "hel" predict "l" next
        - given "hell" predict "o" next

        In addition, because the DataLoader will create batches of examples,
        every forward/backward pass during traning will simultaneously train
        a LOT of predictions, amortizing a lot of computation. In particular,
        for a batched input of integers X (B, T) where B is batch size and
        T is block_size and Y (B, T), the network will during training be
        simultaneously training to make B*T predictions, all at once! Of course,
        at test time we can paralellize across batch B, but unlike during training
        we cannot parallelize across the time dimension T - we have to run
        a forward pass of the network to recover the next single character of the
        sequence along each batch dimension, and repeatedly always feed in a next
        character to get the next one.

        So yes there is a big asymmetry between train/test time of autoregressive
        models. During training we can go B*T at a time with every forward pass,
        but during test time we can only go B at a time, T times, with T forward
        passes.
        """
        x = torch.tensor(dix[:-1], dtype=torch.long)
        y = torch.tensor(dix[1:], dtype=torch.long)
        return x, y


In [ ]:
block_size = 128 # spatial extent of the model for its context

In [ ]:
#!rm -f input.txt*
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
#!curl 'https://gutenberg.org/files/59112/59112-0.txt' |tail -n +292 | head -n 3555 >RUR-Eng.txt

--2026-09-21 07:15:36--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.007s  

2026-09-21 07:15:36 (150 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [ ]:
# you can download this file at https://github.com/karpathy/char-rnn/blob/master/data/tinyshakespeare/input.txt
text = open('input.txt', 'r').read() # don't worry we won't run out of file handles
#text = open('RUR-Eng.txt', 'r').read() # don't worry we won't run out of file handles
train_dataset = CharDataset(text, block_size) # one line of poem is roughly 50 characters

data has 1115394 characters, 65 unique.


In [ ]:
from mingpt.model import GPT, GPTConfig
##@title Model parameters
mconf = GPTConfig(train_dataset.vocab_size, train_dataset.block_size,
                  n_layer=6, n_head=4, n_embd=128
)
model = GPT(mconf)
device = 'cuda'
model.to(device)
model.train()

GPT(
  (tok_emb): Embedding(65, 128)
  (drop): Dropout(p=0.1, inplace=False)
  (blocks): Sequential(
    (0): Block(
      (ln1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      (ln2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      (attn): CausalSelfAttention(
        (key): Linear(in_features=128, out_features=128, bias=True)
        (query): Linear(in_features=128, out_features=128, bias=True)
        (value): Linear(in_features=128, out_features=128, bias=True)
        (attn_drop): Dropout(p=0.1, inplace=False)
        (resid_drop): Dropout(p=0.1, inplace=False)
        (proj): Linear(in_features=128, out_features=128, bias=True)
      )
      (mlp): Sequential(
        (0): Linear(in_features=128, out_features=512, bias=True)
        (1): GELU(approximate='none')
        (2): Linear(in_features=512, out_features=128, bias=True)
        (3): Dropout(p=0.1, inplace=False)
      )
    )
    (1): Block(
      (ln1): LayerNorm((128,), eps=1e-05, elementwise_affi

In [ ]:
from mingpt.trainer import Trainer, TrainerConfig

# initialize a trainer instance and kick off training
tconf = TrainerConfig(max_epochs=2, batch_size=32, learning_rate=6e-4,
                      lr_decay=True, warmup_tokens=512*20, final_tokens=2*len(train_dataset)*block_size,
                      num_workers=0)
trainer = Trainer(model, train_dataset, None, tconf)
trainer.train()

epoch 1 iter 34852: train loss 1.23508. lr 3.000169e-04: 100%|██████████| 34853/34853 [17:47<00:00, 32.64it/s]
epoch 2 iter 24287: train loss 1.02202. lr 6.000000e-05:  70%|██████▉   | 24285/34853 [12:06<05:47, 30.41it/s]

In [ ]:
print('saving model...')
torch.save(model.state_dict(), 'saved_model.pt')
#torch.save(model.state_dict(), 'rur-b512-e2.pt')


saving model...


In [ ]:
model.load_state_dict(torch.load('saved_model.pt'))
#model.load_state_dict(torch.load('rur-b512-e2.pt'))

<All keys matched successfully>

In [ ]:
# alright, let's sample some character-level text
from mingpt.utils import sample

context = "O God, O God!"
#context = "HELENA. How are you?"
#context = "ALQUIST. Go."

x = torch.tensor([train_dataset.stoi[s] for s in context], dtype=torch.long)[None,...].to(device)
y = sample(model, x, 2000, temperature=1.0, sample=True, top_k=10)[0]
completion = ''.join([train_dataset.itos[int(i)] for i in y])
print(completion)

ALQUIST. Go. Gall, Hallemeier, plays Hallemier--_was_ it does them. They’re
waiting for the soul.

DR. GALL. (_Rises_) Yes.

DOMIN. The Robots are no one one other two cooks what the _ound_ of the erhaps.
Why, the old good. (_Rises_) “Dividenten the factory and such a whole shole ship
brought it? What will do, the long like the Robots will not will
starting-now. I wanted to be understand two proposes off bread. And I shall there in
the world.

DR. GALL. No, no. I know, Helena, I’m of you, I’m sure they’d like up.

DR. GALL. (_Seateds his arm_) By Jove, I sy to be understand changes to
mill, Harry. There’s nothing, and I’m not quietion offer keen in to self the
_smallest_. Something lunch least. (_Draws the same away from him, he requirement
we’ve old board your Robots. I will changer the formula--

HALLEMEIER. Primus.

DOMIN. In the charage of us away.

BUSMAN. Well, now, but we’ve one door. I should like to for them, Doctor Gall.

DOMIN. What’s the name. Why?

HELENA. The come of the 

# Tasks

Choose one of the tasts.

**Upload your modified notebook or python script** with results to the [homework vault]( https://nlp.fi.muni.cz/en/NlpInPracticeCourse) (odevzdávárna).

## Task 1

Try several configurations of the model (change number of layers, number of attantion head, embeddings dimensin), investigate the generated texts and describe differences.

In [ ]:
# number of parameters of the model
sum(np.prod(p.size()) for p in model.parameters())


np.int64(1225216)

## Task 2

Implement a new Dataset class (simmilar to the CharDataset class) which uses BPE tokenizer. Describe differences in quality and/or learning time.



In [ ]:
!pip install sentencepiece
import sentencepiece as spm



In [ ]:
#spm.SentencePieceTrainer.train(input='RUR-Eng.txt', model_prefix='rur-spm', vocab_size=200)
spm.SentencePieceTrainer.train(input='input.txt', model_prefix='sha-spm', vocab_size=200)


In [ ]:
!head sha-spm.vocab

<unk>	0
<s>	0
</s>	0
▁	-2.66171
.	-2.85203
e	-3.01347
s	-3.06466
t	-3.37679
o	-3.48077
a	-3.68805


In [ ]:
sp = spm.SentencePieceProcessor(model_file='rur-spm.model')

In [ ]:
sp.encode('HELENA. I thought it was forbidden to--', out_type=str)

['▁HELENA',
 '.',
 '▁I',
 '▁',
 'th',
 'ough',
 't',
 '▁it',
 '▁was',
 '▁for',
 'b',
 'i',
 'd',
 'd',
 'en',
 '▁to',
 '--']

In [ ]:
class SubwordDataset(Dataset):
     pass
